# Семинар 8. Итераторы и генераторы

**Итераторы**
- Итерабельный объект (iterable) - источник данных для итерирования

- Итератор (iterator) - объект-абстракция, извлекающий из источника элемент за элементом и знающий только о том объекте, на котором он в текущий момент остановился

- Итераторы используются для итерирования по последовательностям, при этом:
    - последовательности могут быть неиндексируемыми (например, set)
    - в процессе работы элементы могут фильтроваться или преобразовываться
    - итераторы работают лениво

- В Python для итераторов есть две встроенные функции:
    - iter(x: iterable) - создаёт итератор для переданной последовательности
    - next(x: iterator) - выполняет шаг итерации для переданного итератора

In [4]:
r = iter([1, 2, 3])

print(next(r), next(r), next(r))
#next(r) -> StopIteration

print(type(r))

1 2 3
<class 'list_iterator'>


Функции range и enumerate

In [5]:
r = range(2, 10, 3)
print(type(r))

for e in r:
    print(e, end=' ')

<class 'range'>
2 5 8 

In [10]:
for index, element in enumerate(list('abcdef')):
    print(index, element, end='   ')

0 a   1 b   2 c   3 d   4 e   5 f   

**Итераторы и цикл for**
Цикл for использует итераторы вместо индексов.

In [7]:
# так выглядит
for e in [1, 2, 3]:
    print(e)

1
2
3


In [8]:
# так работает
iterator = iter([1, 2, 3])
while True:
        i = next(iterator)
        print(i)

1
2
3


StopIteration: 

Как он работает на самом деле:

In [ ]:
iterator = iter([1, 2, 3])
while True:
    try:
        i = next(iterator)
        print(i)
    except StopIteration:
        break

**Примеры встроенных функций, возвращающих итераторы:**

- enumerate
- zip
- open
- reversed
- map (похож на генератор, но метод send не реализует)
- filter


# **Генераторы**
Генератор - подтип итератора
- В генераторе есть внутреннее изменяемое состояние в виде локальных переменных, которое он хранит автоматически
- В генератор можно посылать данные между итерациями (метод send)
- Генераторы можно создавать с помощью генераторных выражений или описывать функциями, в которых return заменяется на yield

Пример генератора, полученного с помощью выражения:

In [11]:
gen = (x**2 for x in range(5))

print(next(gen), next(gen))
print(type(gen))

0 1
<class 'generator'>


## Запись списка

Квадратные скобки собирают список сразу.

In [15]:
[x * x for x in range(3)]
# list(x * x for x in range(3))

[0, 1, 4]

**Ключевое слово yield**
- yield - это слово, по смыслу похожее на return
- Но используется в функциях, возвращающих генераторы
- При вызове такой функции тело не выполняется, функция только возвращает генератор
- В первых запуск функция будет выполняться от начала и до yield
- После выхода состояние функции сохраняется
- На следующий вызов будет проводиться итерация цикла и возвращаться следующее значение
- И так далее, пока не кончится цикл каждого yield в теле функции
- После этого генератор станет пустым

In [42]:
def sample_yield():
    print("Делаю нечто 1")
    yield 1
    print("Делаю нечто 2")
    print("Делаю нечто 3")
    yield 1
    print("Делаю нечто 4")
    yield 2

g = sample_yield()

print(next(g))
print(next(g))
print(next(g))

# print(next(g))

Делаю нечто 1
1
Делаю нечто 2
Делаю нечто 3
1
Делаю нечто 4
2
Делаю нечто 1
1
Делаю нечто 2
Делаю нечто 3
1
Делаю нечто 4
2


In [43]:
sample_yield_generator = sample_yield()
for x in sample_yield_generator:
    print(x)

Делаю нечто 1
1
Делаю нечто 2
Делаю нечто 3
1
Делаю нечто 4
2


In [13]:
def up_to(n):
    for i in range(n):
        yield i

gen = up_to(4)
print(next(gen))
print(next(gen))
print(next(gen))
print(next(gen))


0
1
2
3


In [14]:
# значения закончились
print(next(gen))

StopIteration: 

## Круглые скобки

Тот же вид объекта, что у функции с `yield`. Список не собран.

In [19]:
squares_expr = (x * x for x in range(3))
print(next(squares_expr))
print(next(squares_expr))


0
1


## Задача

Функция `squares(n)`. Отдать квадраты `0`, `1`, `4`, …, `(n - 1)²`.

`list(squares(4))` даёт `[0, 1, 4, 9]`.

Генераторы и метод send
- Выражение yield x выполняет две вещи:
    - Возвращает вызывающему генератор коду значение x
    - Возвращает на уровне кода генератора значение None
- Расширенный протокол генератора содержит метод send, получающий на вход произвольный объект y
- В случае вызова send(y) выполняется то же, что и при next, но yield вернет на уровне кода генератора не None, а y


In [40]:
def create_gen():
    y = 1
    for x in range(5):
        y = yield x
        yield x
        print("y=%s" %y)

gen = create_gen()
print(next(gen))
print(next(gen))
print(gen.send(10))
print(next(gen))


0
0
y=None
1
1
y=<list_iterator object at 0x0000022D96173040>
2


**Итераторы и функция range**
Результат работы range не является итератором, хотя и выполняет его роль:

In [21]:
print('__next__' in dir(zip([], [])))
print('__next__' in dir(range(3)))
# print(dir(range(3)))

True
False


Особенности объектов range:
- как и итераторы, выполняются лениво
- являются неизменяемыми (могут быть ключами словаря)
- имеют полезные атрибуты (len, index, __getitem__)
- по ним можно итерироваться многократно

In [22]:
r = range(3)
for i in r:
    print(i)
print("---------------------")
for i in r:
    print(i)

0
1
2
---------------------
0
1
2


## Декоратор

Декораторы — это функциональные конструкции, позволяющие расширить возможности функцию без изменения исходного кода.
Декораторы опираются на три ключевых концепции Python:

- В Python функции являются объектами первого класса — их можно передавать как аргументы
- Функции могут быть определены внутри других функций
- Функция может возвращать другую функцию

Определим декоратор.

`my_decorator` принимает функцию и возвращает `wrapper`.

In [23]:
def my_decorator(func):
    def wrapper():
        print("Действия перед вызовом функции")
        func()
        print("Действия после вызова функции")
    return wrapper

Определим функцию

In [24]:
def say_hello():
    print("Привет!")

say_hello()

Привет!


Изменим поведение функции не меняя саму функцию с помощью декоратора.

Строка `@my_decorator` — это запись `inc = my_decorator(say_hello)`.

In [25]:
@my_decorator
def say_hello():
    print("Привет!")
say_hello()

Действия перед вызовом функции
Привет!
Действия после вызова функции


## `itertools`

`count()` отдаёт `0`, `1`, `2`, … и сам не кончается. `islice` забирает первые четыре. `accumulate` копит сумму от начала списка.


In [26]:
from itertools import count, islice, accumulate

print(list(islice(count(), 4)))


[0, 1, 2, 3]


In [27]:
print(list(accumulate([2, 2, 2, 2])))

[2, 4, 6, 8]


## Задачи для самоконтроля


### 1. `iter` и `next`

`it = iter([10, 20, 30])`. Что вернут первые три `next`? Как называется остановка на четвёртом?


### 2. `range`

`r = range(3)`. Что сделает `next(r)`? Что вернёт `next(iter(r))`? Что напечатают два цикла `for` по одному и тому же `r`?


### 3. `yield`

Написать `powers_of_two(k)`. Функция отдаёт `k` степеней двойки, начиная с `1`. `list(powers_of_two(4))` равно `[1, 2, 4, 8]`.

Чему равны первые два `next` от `(x * x for x in range(4))`?


### 4. `send`

Дана функция.

```python
def gen():
    for x in range(4):
        y = yield x
```

Заполнить таблицу для вызовов `next`, `next`, `send(7)`, `next`. В каждой строке: что отдано наружу и чему равно `y` после продолжения.


### 5. Декоратор

Написать `twice`. Обёртка возвращает `func(func(x))`. Через `@` повесить `twice` на функцию `x + 1`. Чему равен вызов от `3`?


### 6. `itertools`

Собрать из `count` и `islice` первые пять кратных трёх: `[0, 3, 6, 9, 12]`.

Руками выписать `accumulate([1, 2, 3, 4])`.


### 7. Три блока вместе

Одна программа.

Функция `odds` бесконечно отдаёт нечётные `1`, `3`, `5`, `7`, …. Счётчик свой, без `range`.

Декоратор `take_four` принимает функцию и возвращает обёртку. Обёртка вызывает эту функцию и возвращает `list(islice(..., 4))`.

Над `odds` стоит `@take_four`. Чему равен вызов `odds()`?
